# HW5 – Altair + Jekyll

Workbook for building_inventory.csv

In [10]:
import pandas as pd
import altair as alt

# Allow large datasets for Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Load Data

In [11]:
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"
df = pd.read_csv(url)

df.head(), df.columns

(                       Agency Name  \
 0  Department of Natural Resources   
 1  Department of Natural Resources   
 2  Department of Natural Resources   
 3  Department of Natural Resources   
 4  Department of Natural Resources   
 
                                      Location Name             Address  \
 0  Anderson Lake Conservation Area - Fulton County  Anderson Lake C.a.   
 1  Anderson Lake Conservation Area - Fulton County  Anderson Lake C.a.   
 2  Anderson Lake Conservation Area - Fulton County  Anderson Lake C.a.   
 3  Anderson Lake Conservation Area - Fulton County  Anderson Lake C.a.   
 4  Anderson Lake Conservation Area - Fulton County  Anderson Lake C.a.   
 
       City  Zip code  County  Congress Dist Congressional Full Name  Rep Dist  \
 0  Astoria     61501  Fulton             17            Cheri Bustos        93   
 1  Astoria     61501  Fulton             17            Cheri Bustos        93   
 2  Astoria     61501  Fulton             17            Cheri Bust

## Clean and Transform Data

In [12]:
# Adjust these names if needed after checking df.columns
sqft_col = "Square Footage"
agency_col = "Agency Name"
year_col = "Year Constructed"
usage_col = "Usage Description"

# Convert numeric columns
df[sqft_col] = pd.to_numeric(df[sqft_col], errors="coerce")
df[year_col] = pd.to_numeric(df[year_col], errors="coerce")

# Drop rows with missing or zero square footage
df_clean = df.dropna(subset=[sqft_col]).copy()
df_clean = df_clean[df_clean[sqft_col] > 0]

# Keep reasonable years
df_clean = df_clean[(df_clean[year_col] >= 1800) & (df_clean[year_col] <= 2025)]

df_clean.head()

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


## Plot 1 – Top Agencies by Total Square Footage

In [13]:
# Aggregate total square footage by agency
agency_sqft = (
    df_clean
    .groupby(agency_col, as_index=False)[sqft_col]
    .sum()
    .rename(columns={sqft_col: "total_sqft"})
)

# Top 10 agencies
top10_agencies = agency_sqft.nlargest(10, "total_sqft")
top10_agencies

,Agency Name,total_sqft
33,University of Illinois,25016006
7,Department of Corrections,15104759
32,Southern Illinois University,8705319
8,Department of Human Services,8465976
15,Department of Transportation,5585362
6,Department of Central Management Services,4260911
10,Department of Military Affairs,4077299
11,Department of Natural Resources,3843987
29,Northern Illinois University,3667294
27,Illinois State University,2960272


In [14]:
plot1 = (
    alt.Chart(top10_agencies)
    .mark_bar()
    .encode(
        x=alt.X("total_sqft:Q", title="Total square footage"),
        y=alt.Y(f"{agency_col}:N", sort="-x", title="Agency"),
        color=alt.Color("total_sqft:Q", title="Total square footage", scale=alt.Scale(scheme="blues")),
        tooltip=[
            alt.Tooltip(agency_col, title="Agency"),
            alt.Tooltip("total_sqft:Q", title="Total sq ft", format=",")
        ]
    )
    .properties(
        title="Top 10 Agencies by Total Building Square Footage",
        width=650,
        height=350
    )
)

plot1

alt.Chart(...)

In [15]:
plot1.save("plot1.json")

## Plot 2 – Interactive Decade Trend by Usage Description

In [16]:
# Create decade column
df_clean["decade"] = (df_clean[year_col] // 10) * 10

# Aggregate by decade and usage description
decade_usage = (
    df_clean
    .groupby(["decade", usage_col], as_index=False)[sqft_col]
    .sum()
    .rename(columns={sqft_col: "total_sqft"})
)

# Identify top usage categories by total square footage
usage_totals = (
    decade_usage
    .groupby(usage_col, as_index=False)["total_sqft"]
    .sum()
    .sort_values("total_sqft", ascending=False)
)

TOP_N = 8
top_usage_names = usage_totals.head(TOP_N)[usage_col].tolist()

decade_usage_top = decade_usage[decade_usage[usage_col].isin(top_usage_names)].copy()
decade_usage_top.head()

,decade,Usage Description,total_sqft
0,1800,Assembly,4440
1,1810,Assembly,9533
2,1810,Education,800
3,1830,Assembly,278054
4,1830,Residential,3123


In [17]:
# Param-based dropdown for usage description (Altair 5)
usage_param = alt.param(
    name="usage_param",
    bind=alt.binding_select(options=top_usage_names, name="Usage: "),
    value=top_usage_names[0]
)

plot2 = (
    alt.Chart(decade_usage_top)
    .mark_bar()
    .encode(
        x=alt.X("decade:O", title="Construction decade"),
        y=alt.Y("total_sqft:Q", title="Total square footage"),
        color=alt.Color("decade:O", title="Decade"),
        tooltip=[
            alt.Tooltip("decade:O", title="Decade"),
            alt.Tooltip(usage_col, title="Usage"),
            alt.Tooltip("total_sqft:Q", title="Total sq ft", format=",")
        ]
    )
    .add_params(usage_param)
    # Filter expression ties the param value to the usage column
    .transform_filter("datum['{}'] == usage_param".format(usage_col))
    .properties(
        title="Total Building Square Footage by Construction Decade (Interactive by Usage)",
        width=650,
        height=350
    )
)

plot2

alt.Chart(...)

In [18]:
plot2.save("plot2.json")